## Task 3.1: Ensure Data Consistency Across State and District Levels

In [2]:
import pandas as pd
import numpy as np

In [5]:
file_path = "../data/phonepe-pulse_raw-data_q12018-to-q22021-v0-1-5-1720351752.xlsx"

state_txn_users = pd.read_excel(file_path, sheet_name="State_Txn and Users")

district_txn_users = pd.read_excel(file_path, sheet_name="District_Txn and Users")

print("Datasets loaded successfully!")

Datasets loaded successfully!


## Task 3.1: Ensure Data Consistency Across State and District Levels

In [7]:
district_summary = (
    district_txn_users
    .groupby(["State", "Year", "Quarter"])[
        ["Transactions", "Amount (INR)", "Registered Users"]
    ]
    .sum()
    .reset_index()
)

district_summary.head()   # Calculate State-Level Totals from District Data

,State,Year,Quarter,Transactions,Amount (INR),Registered Users
0,Andaman & Nicobar Islands,2018,1,6658,1.463176e+07,6740
1,Andaman & Nicobar Islands,2018,2,11340,2.833854e+07,9405
2,Andaman & Nicobar Islands,2018,3,16265,5.555747e+07,12149
3,Andaman & Nicobar Islands,2018,4,23758,9.054834e+07,15222
4,Andaman & Nicobar Islands,2019,1,30486,1.022997e+08,18596


In [9]:
state_summary = state_txn_users[
    [
        "State",
        "Year",
        "Quarter",
        "Transactions",
        "Amount (INR)",
        "Registered Users"
    ]
]   # Extract Required Columns from State Dataset

In [ ]:
district_summary = district_summary.rename(
    columns={
        "Transactions": "District_Transactions",
        "Amount (INR)": "District_Amount",
        "Registered Users": "District_Users"
    }
)

state_summary = state_summary.rename(
    columns={
        "Transactions": "State_Transactions",
        "Amount (INR)": "State_Amount",
        "Registered Users": "State_Users"
    }
)  # Rename Columns for Easy Comparison

In [12]:
comparison = pd.merge(
    state_summary,
    district_summary,
    on=["State", "Year", "Quarter"],
    how="inner"
)

comparison.head()    # Merge State and District Summaries

,State,Year,Quarter,State_Transactions,State_Amount,State_Users,District_Transactions,District_Amount,District_Users
0,Andaman & Nicobar Islands,2018,1,6658,1.463176e+07,6740,6658,1.463176e+07,6740
1,Andaman & Nicobar Islands,2018,2,11340,2.833854e+07,9405,11340,2.833854e+07,9405
2,Andaman & Nicobar Islands,2018,3,16265,5.555747e+07,12149,16265,5.555747e+07,12149
3,Andaman & Nicobar Islands,2018,4,23758,9.054834e+07,15222,23758,9.054834e+07,15222
4,Andaman & Nicobar Islands,2019,1,30486,1.022997e+08,18596,30486,1.022997e+08,18596


In [14]:
comparison["Transaction_Difference"] = (
    comparison["State_Transactions"]
    - comparison["District_Transactions"]
)

comparison["Amount_Difference"] = (
    comparison["State_Amount"]
    - comparison["District_Amount"]
)

comparison["User_Difference"] = (
    comparison["State_Users"]
    - comparison["District_Users"]
)

comparison.head()

,State,Year,Quarter,State_Transactions,State_Amount,State_Users,District_Transactions,District_Amount,District_Users,Transaction_Difference,Amount_Difference,User_Difference
0,Andaman & Nicobar Islands,2018,1,6658,1.463176e+07,6740,6658,1.463176e+07,6740,0,-7.636845e-08,0
1,Andaman & Nicobar Islands,2018,2,11340,2.833854e+07,9405,11340,2.833854e+07,9405,0,-1.490116e-08,0
2,Andaman & Nicobar Islands,2018,3,16265,5.555747e+07,12149,16265,5.555747e+07,12149,0,7.450581e-09,0
3,Andaman & Nicobar Islands,2018,4,23758,9.054834e+07,15222,23758,9.054834e+07,15222,0,-1.043081e-07,0
4,Andaman & Nicobar Islands,2019,1,30486,1.022997e+08,18596,30486,1.022997e+08,18596,0,0.000000e+00,0


In [15]:
discrepancies = comparison[
    (comparison["Transaction_Difference"] != 0)
    |
    (comparison["Amount_Difference"] != 0)
    |
    (comparison["User_Difference"] != 0)
]

discrepancies

,State,Year,Quarter,State_Transactions,State_Amount,State_Users,District_Transactions,District_Amount,District_Users,Transaction_Difference,Amount_Difference,User_Difference
0,Andaman & Nicobar Islands,2018,1,6658,1.463176e+07,6740,6658,1.463176e+07,6740,0,-7.636845e-08,0
1,Andaman & Nicobar Islands,2018,2,11340,2.833854e+07,9405,11340,2.833854e+07,9405,0,-1.490116e-08,0
2,Andaman & Nicobar Islands,2018,3,16265,5.555747e+07,12149,16265,5.555747e+07,12149,0,7.450581e-09,0
3,Andaman & Nicobar Islands,2018,4,23758,9.054834e+07,15222,23758,9.054834e+07,15222,0,-1.043081e-07,0
5,Andaman & Nicobar Islands,2019,2,33689,1.202547e+08,21731,33689,1.202547e+08,21731,0,1.639128e-07,0
...,...,...,...,...,...,...,...,...,...,...,...,...
499,West Bengal,2020,2,57676797,1.000994e+11,13222022,57676797,1.000994e+11,13222022,0,8.697510e-04,0
500,West Bengal,2020,3,79954504,1.568134e+11,14448366,79954504,1.568134e+11,14448366,0,1.525879e-04,0
501,West Bengal,2020,4,100340645,1.991655e+11,15662093,100340645,1.991655e+11,15662093,0,-1.068115e-03,0
502,West Bengal,2021,1,118254052,2.429372e+11,16808799,118254052,2.429372e+11,16808799,0,-9.155273e-04,0


In [16]:
print("Total discrepancies found:")

print(len(discrepancies))

Total discrepancies found:
491


In [17]:
discrepancies[
    [
        "State",
        "Year",
        "Quarter",
        "Transaction_Difference",
        "Amount_Difference",
        "User_Difference"
    ]
]

,State,Year,Quarter,Transaction_Difference,Amount_Difference,User_Difference
0,Andaman & Nicobar Islands,2018,1,0,-7.636845e-08,0
1,Andaman & Nicobar Islands,2018,2,0,-1.490116e-08,0
2,Andaman & Nicobar Islands,2018,3,0,7.450581e-09,0
3,Andaman & Nicobar Islands,2018,4,0,-1.043081e-07,0
5,Andaman & Nicobar Islands,2019,2,0,1.639128e-07,0
...,...,...,...,...,...,...
499,West Bengal,2020,2,0,8.697510e-04,0
500,West Bengal,2020,3,0,1.525879e-04,0
501,West Bengal,2020,4,0,-1.068115e-03,0
502,West Bengal,2021,1,0,-9.155273e-04,0


# Conclusion

Task 3 completed successfully.

- Aggregated district-level data to state level.
- Compared transactions, amount, and registered users.
- Identified mismatches between datasets.
- Displayed all discrepancies for further investigation.